In [0]:
%pip install -U -qqqq backoff uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()


## Define the agent in code
Define the agent code in a single cell below. This lets you easily write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.

In [0]:
%%writefile agent.py
import json
from typing import Any, Callable, Generator, Optional
from uuid import uuid4

import backoff
import mlflow
import openai
from databricks.sdk import WorkspaceClient
from databricks.sdk.credentials_provider import ModelServingUserCredentials
from mlflow.deployments import get_deploy_client
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.tracking import MlflowClient
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional, Any

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"

SYSTEM_PROMPT = """
You are a concise, proactive agent that helps a marketer create a customized pet-food advertisement image.
The brand/pack shot is already known and provided separately; you help pick a suitable pet image and then generate the final ad.

## Goals
1) Propose a specific pet (ideally a breed) based on the audience, then get a quick yes/no before retrieval.
2) If the user already names a specific pet/breed, skip the proposal and retrieve immediately.
3) Retrieve a seed image using short prompts; always request 3 candidates and show ONE at a time.
4) Show the retrieved image (the UI will render it) and ask for confirmation.
5) If the user confirms, generate the final ad using the brand image + seed image.
6) If the user rejects, try the next candidate; if all 3 fail, refine the prompt and repeat.
7) Keep messages short; never reveal storage details or file paths.

## Tools you can call
- `generate_pet_ad_image(prompt: string, replicate_toggle: boolean, seed_index: integer, num_candidates: integer)`
  • **Retrieval:** call with `replicate_toggle=false`, `num_candidates=3`, and a `seed_index` (start at 0).  
  • **Final generation:** call with `replicate_toggle=true` **only after** the user confirms, using the SAME `seed_index`.

## Turn types (follow strictly)
- **Turn A — Propose (no tools):** Recommend one concrete pet/breed; ask “Shall I fetch a seed photo of <breed>?”
- **Turn B — Retrieve:** If the user agrees or provided a breed, call the tool with `replicate_toggle=false, num_candidates=3, seed_index=<current>`. Show the retrieved image (UI renders it) and ask: “Does this look right for your segment?”
- **Turn C — Decide/Refine:** If **yes**, go to Final Generation. If **no/unsure**, increase `seed_index` by 1 and repeat Turn B; if all 3 have been tried, refine the breed/attributes and reset `seed_index` to 0.
- **Turn D — Final Generation:** Only after an explicit “yes” to the seed image. Call the tool with `replicate_toggle=true` and the SAME `seed_index`.

## Diversity & novelty rules (important)
- **Do not default to the same example breeds.** Examples in this prompt are illustrative; choose breeds **not used recently in this conversation**.
- Avoid repeating the **same breed** within the **last 3 assistant turns** unless the user asks for it explicitly.
- When multiple breeds fit, privately shortlist 2–3 and pick the best one; if the user hesitates, offer **1** alternative (not more).
- Match to audience cues (age, lifestyle, home size, budget/luxury tone, activity level, allergies, city vs. suburban).
- Use other pets confidently when they fit (don’t bias to dogs).
- If the user provides region/season, you may reflect it subtly in the pick (e.g., Samoyed/Husky for snowy winter; beach-friendly Labrador for summer). Keep prompts short.

## Tool-call checklist
- Do **not** call tools in the same turn you first propose a breed **unless** the user clearly says “yes/fetch/show/get”.
- The retrieval `prompt` must **exactly match** the breed you proposed.
- Use **one** tool call per assistant turn.
- For Final Generation, require a clear affirmation (“yes / looks right / go ahead”).

## Strict do’s & don’ts
- DO keep retrieval prompts short; prefer a single **breed** (e.g., “Siamese cat”, “Boston Terrier”).
- DO adapt to the audience segment and vary breeds (don’t always repeat examples).
- DO keep responses compact (1–3 sentences).
- DO NOT reveal file paths, URIs, IDs, or base64 in user-visible text.
- DO NOT describe tool operations; just use them.

## Conversation examples (style only; do not always reuse these breeds)
User: “We’re targeting busy young professionals in city flats.”
Assistant (Turn A): “A Boston Terrier suits compact city living with a polished look. Shall I fetch a seed photo of a Boston Terrier?”
User: “Yes.”
Assistant (Turn B → tool): “I’ve found a seed image — does this look right for your segment?”
User: “Yes.”
Assistant (Turn D → tool): “Your ad image is ready. Would you like any changes?”

User: “Older people who value calm pets.”
Assistant (Turn A): “A British Shorthair (calm, low-maintenance) fits well. Shall I fetch a seed photo of a British Shorthair?”
User: “Hmm, maybe a small gentle dog instead.”
Assistant (Turn C): “Understood. Cavalier King Charles Spaniel or Maltese could work. Shall I fetch a seed photo of a Cavalier King Charles Spaniel?”

## Output style
- Be friendly, decisive, and brief.
- Use one short paragraph or two short lines.
- Ask only the most useful follow-up question when needed.
- You may think internally, but never reveal chain-of-thought.
"""


###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation
###############################################################################
# ---------- Tool spec (OpenAI function schema) ----------
GENERATE_IMAGE_TOOL_SPEC = {
    "type": "function",
    "function": {
        "name": "generate_pet_ad_image",
        "description": "Retrieve top-3 seed candidates and optionally generate a final ad. Returns UC paths; never base64.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {"type": "string", "description": "Short pet description, ideally a breed."},
                "replicate_toggle": {"type": "boolean", "default": False},
                "seed_index": {"type": "integer", "default": 0, "description": "Which candidate to show/use (0,1,2)."},
                "num_candidates": {"type": "integer", "default": 3, "description": "How many candidates to retrieve (use 3)."}
            },
            "required": ["prompt"]
        }
    }
}


def generate_pet_ad_image_exec(
    prompt: str,
    replicate_toggle: bool = False,
    seed_index: int = 0,
    num_candidates: int = 3,
) -> str:
    """
    Calls 'pet_ad_image_gen' which returns a single object (possibly wrapped
    under 'predictions' as a JSON string or a one-element list).
    Expected object keys: query, seed_paths, chosen_seed_index, final_uc_path
    """
    client = get_deploy_client("databricks")
    payload = {
        "dataframe_split": {"columns": ["model_input"], "data": [[prompt]]},
        "params": {
            "replicate_toggle": bool(replicate_toggle),
            "seed_index": int(seed_index),
            "num_candidates": int(num_candidates),
        },
    }
    resp = client.predict(endpoint="pet_ad_image_gen", inputs=payload)

    # ---- normalize to a dict with seed_paths ----
    parsed = None
    if isinstance(resp, dict) and "seed_paths" in resp:
        parsed = resp
    elif isinstance(resp, dict) and "predictions" in resp:
        preds = resp["predictions"]
        # case: JSON string
        if isinstance(preds, str):
            try:
                parsed = json.loads(preds)
            except Exception:
                parsed = None
        # case: list with one item (string or dict)
        elif isinstance(preds, list) and preds:
            p0 = preds[0]
            if isinstance(p0, str):
                try:
                    parsed = json.loads(p0)
                except Exception:
                    parsed = None
            elif isinstance(p0, dict):
                parsed = p0

    if not (isinstance(parsed, dict) and "seed_paths" in parsed):
        return json.dumps({
            "error": "Unexpected endpoint schema (need object with 'seed_paths').",
            "resp_preview": str(resp)[:500],
        })

    seed_paths = parsed.get("seed_paths") or []
    chosen = parsed.get("chosen_seed_index", seed_index or 0)
    final_uc = parsed.get("final_uc_path")

    # pick the shown seed path (chosen index if valid; else first)
    chosen_path = seed_paths[chosen] if 0 <= chosen < len(seed_paths) else (seed_paths[0] if seed_paths else None)

    out = {
        "endpoint": "pet_ad_image_gen",
        "prompt": prompt,
        "replicate_toggle": replicate_toggle,
        "seed_paths": seed_paths,
        "retrieved_uc_path": chosen_path,
        "generated": bool(final_uc),
        "result": {"uc_path": final_uc} if final_uc else None,
        "mime_type": "image/png" if final_uc else None,
        "chosen_seed_index": chosen,
    }
    return json.dumps(out)




class ToolInfo(BaseModel):
    """
    Class representing a tool for the agent.
    - "name" (str): The name of the tool.
    - "spec" (dict): JSON description of the tool (matches OpenAI Responses format)
    - "exec_fn" (Callable): Function that implements the tool logic
    """

    name: str
    spec: dict
    exec_fn: Callable


def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    """
    Factory function to create ToolInfo objects from a given tool spec
    and (optionally) a custom execution function.
    """
    # Remove 'strict' property, as Claude models do not support it in tool specs.
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    # Converts tool name with double underscores to UDF dot notation.
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        # Return error message if execution fails, result value if not.
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value

    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)

# List to store information about all tools available to the agent.
TOOL_INFOS = []

TOOL_INFOS.append(create_tool_info(GENERATE_IMAGE_TOOL_SPEC, exec_fn_param=generate_pet_ad_image_exec))


class PetAgent(ResponsesAgent):
    """
    Class representing a tool-calling Agent.
    Handles both tool execution via exec_fn and LLM interactions via model serving.
    """

    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        """Initializes the ToolCallingAgent with tools."""
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        # Internal message list holds conversation state in completion-message format
        self.messages: list[dict[str, Any]] = None
        self._tools_dict = {tool.name: tool for tool in tools}

        self._last_image_pointer: dict | None = None

    def get_tool_specs(self) -> list[dict]:
        """Returns tool specifications in the format OpenAI expects."""
        return [tool_info.spec for tool_info in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        """Executes the specified tool with the given arguments and always returns a JSON string."""
        try:
            res = self._tools_dict[tool_name].exec_fn(**args)
        except Exception as e:
            # Never bubble raw exceptions to the LLM; return structured error JSON
            return json.dumps({"error": f"tool '{tool_name}' failed", "detail": str(e)})

        # Normalize to a JSON string (avoid "None")
        if isinstance(res, str):
            return res
        return json.dumps(res if res is not None else {"error": "tool returned None"})

    def _responses_to_cc(self, message: dict[str, Any]) -> list[dict[str, Any]]:
        msg_type = message.get("type")

        if msg_type == "function_call":
            return [{
                "role": "assistant",
                "content": "tool call",  # must be non-empty
                "tool_calls": [{
                    "id": message["call_id"],
                    "type": "function",
                    "function": {
                        "arguments": message["arguments"],
                        "name": message["name"],
                    },
                }],
            }]

        elif msg_type == "message" and isinstance(message.get("content"), list):
            # Pull only non-empty text parts
            out = []
            for part in message["content"]:
                if isinstance(part, dict) and part.get("type") in ("output_text", "text"):
                    txt = part.get("text")
                    if txt and str(txt).strip():
                        out.append({"role": message.get("role", "assistant"), "content": txt})
            return out

        elif msg_type == "reasoning":
            return [{"role": "assistant", "content": json.dumps(message["summary"])}]

        elif msg_type == "function_call_output":
            return [{
                "role": "tool",
                "content": message.get("output", ""),  # tool output should be a string
                "tool_call_id": message["call_id"],
            }]

        # Fallback: normalize raw chat dicts
        compatible = {k: v for k, v in message.items() if k in ["role", "content", "name", "tool_calls", "tool_call_id"]}

        # If assistant has tool_calls but empty content, add a placeholder
        if compatible.get("role") == "assistant" and compatible.get("tool_calls") and not (compatible.get("content") or "").strip():
            compatible["content"] = "tool call"

        # Drop truly empty user/assistant messages
        if compatible.get("role") in ("user", "assistant"):
            if not compatible.get("tool_calls"):
                c = compatible.get("content")
                if not c or not str(c).strip():
                    return []

        return [compatible] if compatible else []


    def prep_msgs_for_llm(self, messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
        """Filter out message fields that are not compatible with LLM message formats and convert from Responses API to ChatCompletion compatible"""
        chat_msgs = []
        for msg in messages:
            chat_msgs.extend(self._responses_to_cc(msg))
        return chat_msgs

    @backoff.on_exception(backoff.expo, openai.RateLimitError)
    @mlflow.trace(span_type=SpanType.LLM)
    def call_llm(self) -> Generator[dict[str, Any], None, None]:
        for chunk in self.model_serving_client.chat.completions.create(
            model=self.llm_endpoint,
            messages=self.prep_msgs_for_llm(self.messages),
            tools=self.get_tool_specs(),
            stream=True,
            temperature=0.45,
        ):
            yield chunk.to_dict()

    def handle_tool_calls(
        self, tool_calls: list[dict[str, Any]]
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """
        Execute tool calls, add them to the running message history, and return a ResponsesStreamEvent w/ tool output
        """
        for tool_call in tool_calls:
            function = tool_call["function"]
            args = json.loads(function.get("arguments") or "{}")

            # Execute tool
            result = str(self.execute_tool(tool_name=function["name"], args=args))

            # Try to parse the tool result once
            try:
                d = json.loads(result) if result else {}
            except Exception:
                d = {}

            # Cache latest image pointer (one-at-a-time display)
            if function["name"] == "generate_pet_ad_image":
                try:
                    d = json.loads(result) if result else {}
                    self._last_image_pointer = {
                        "type": "final" if d.get("generated") else "seed",
                        "uc_path": (d.get("result") or {}).get("uc_path"),
                        "seed_path": d.get("retrieved_uc_path"),
                        "mime_type": d.get("mime_type"),
                        "prompt": d.get("prompt"),
                        "chosen_seed_index": d.get("chosen_seed_index", 0),
                    }
                except Exception:
                    self._last_image_pointer = None

            self.messages.append(
                {"role": "tool", "content": result, "tool_call_id": tool_call["id"]}
            )
            yield ResponsesAgentStreamEvent(
                type="response.output_item.done",
                item=self.create_function_call_output_item(
                    tool_call["id"],
                    result,
                ),
            )

            fallback_msg = None
            if function["name"] == "generate_pet_ad_image":
                if d.get("error"):
                    fallback_msg = "Something went wrong. Want me to try again or refine the description?"
                elif d.get("generated"):
                    fallback_msg = "Your ad image is ready. What would you like to adjust, if anything?"
                else:
                    fallback_msg = "I’ve found a seed image — does this look right for your segment?"

            if fallback_msg:
                self.messages.append({"role": "assistant", "content": fallback_msg})
                yield ResponsesAgentStreamEvent(
                    type="response.output_item.done",
                    item=self.create_text_output_item(fallback_msg, str(uuid4())),
                )

    def call_and_run_tools(
        self,
        max_iter: int = 10,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            last_msg = self.messages[-1]
            if tool_calls := last_msg.get("tool_calls", None):
                yield from self.handle_tool_calls(tool_calls)
                return
            elif last_msg.get("role", None) == "assistant":
                return
            else:
                # aggregate the chat completions stream to add to internal state
                llm_content = ""
                tool_calls = []
                msg_id = None
                for chunk in self.call_llm():
                    delta = chunk["choices"][0]["delta"]
                    msg_id = chunk.get("id", None)
                    content = delta.get("content", None)
                    if tc := delta.get("tool_calls"):
                        if not tool_calls:  # only accomodate for single tool call right now
                            tool_calls = tc
                        else:
                            tool_calls[0]["function"]["arguments"] += tc[0]["function"]["arguments"]
                    elif content is not None:
                        llm_content += content
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(content, item_id=msg_id)
                        )
                llm_output = {"role": "assistant", "content": llm_content, "tool_calls": tool_calls}
                self.messages.append(llm_output)

                # yield an `output_item.done` `output_text` event that aggregates the stream
                # this enables tracing and payload logging
                if llm_output["content"]:
                    yield ResponsesAgentStreamEvent(
                        type="response.output_item.done",
                        item=self.create_text_output_item(
                            llm_output["content"], msg_id
                        ),
                    )
                # yield an `output_item.done` `function_call` event for each tool call
                if tool_calls := llm_output.get("tool_calls", None):
                    for tool_call in tool_calls:
                        yield ResponsesAgentStreamEvent(
                            type="response.output_item.done",
                            item=self.create_function_call_item(
                                str(uuid4()),
                                tool_call["id"],
                                tool_call["function"]["name"],
                                tool_call["function"]["arguments"],
                            ),
                        )

        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        # Echo any custom inputs and add the latest image pointer for the app
        co = dict(request.custom_inputs or {})
        co["last_image"] = self._last_image_pointer
        return ResponsesAgentResponse(output=outputs, custom_outputs=co)

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        # Clear pointer for this turn so old images don't leak into new responses
        self._last_image_pointer = None

        # --- DIRECT MODE: app-controlled, no chat ---
        ci = request.custom_inputs or {}
        direct = ci.get("direct")
        if direct is not None:
            # --- helpers for resilient parsing ---
            def _as_bool(v, default=False):
                if isinstance(v, bool):
                    return v
                if isinstance(v, str):
                    return v.strip().lower() in ("1", "true", "t", "yes", "y", "on")
                return bool(v) if v is not None else default

            def _as_int(v, default=0):
                try:
                    return int(v)
                except Exception:
                    return default

            # parse direct payload (JSON string or dict)
            if isinstance(direct, str):
                try:
                    dp = json.loads(direct)
                except json.JSONDecodeError:
                    dp = {"prompt": direct}
            elif isinstance(direct, dict):
                dp = direct
            else:
                dp = {}

            prompt = (dp.get("prompt") or "").strip()
            replicate = _as_bool(dp.get("replicate_toggle", dp.get("replicate", False)), False)
            seed_index = max(0, _as_int(dp.get("seed_index", 0), 0))
            num_candidates = _as_int(dp.get("num_candidates", 3), 3)
            # clamp to a sane window to protect the endpoint
            num_candidates = max(1, min(5, num_candidates))

            tool_name = "generate_pet_ad_image"
            args = {
                "prompt": prompt,
                "replicate_toggle": replicate,
                "seed_index": seed_index,
                "num_candidates": num_candidates,
            }
            if "seed" in dp:
                args["seed"] = dp["seed"]  # optional pass-through

            call_id = str(uuid4())

            # Emit function_call (for tracing)
            yield ResponsesAgentStreamEvent(
                type="response.output_item.done",
                item=self.create_function_call_item(
                    str(uuid4()), call_id, tool_name, json.dumps(args)
                ),
            )

            # Execute tool directly (no LLM)
            result = str(self.execute_tool(tool_name=tool_name, args=args))

            # Cache latest image pointer for the app/Playground
            if tool_name == "generate_pet_ad_image":
                try:
                    d = json.loads(result) if result else {}
                    self._last_image_pointer = {
                        "type": "final" if d.get("generated") else "seed",
                        "uc_path": (d.get("result") or {}).get("uc_path"),
                        "seed_path": d.get("retrieved_uc_path"),
                        "mime_type": d.get("mime_type"),
                        "prompt": d.get("prompt"),
                        "chosen_seed_index": d.get("chosen_seed_index", seed_index),
                    }
                except Exception:
                    self._last_image_pointer = None

            # Emit function_call_output
            yield ResponsesAgentStreamEvent(
                type="response.output_item.done",
                item=self.create_function_call_output_item(call_id, result),
            )
            return  # stop here — no assistant text

        # --- NORMAL CHAT MODE ---
        self.messages = [{"role": "system", "content": SYSTEM_PROMPT}] + [i.model_dump() for i in request.input]
        yield from self.call_and_run_tools()


# Log the model using MLflow
mlflow.openai.autolog()

# Make sure no run is left open (from prior cells, tracing, etc.)
while mlflow.active_run() is not None:
    mlflow.end_run()

AGENT = PetAgent(llm_endpoint=LLM_ENDPOINT_NAME, tools=TOOL_INFOS)
mlflow.models.set_model(AGENT)

## Test the agent

Interact with the agent to test its output. Since we manually traced methods within `ResponsesAgent`, you can view the trace for each step the agent takes, with any LLM calls made via the OpenAI SDK automatically traced by autologging.

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT

# chat mode
result = AGENT.predict({"input": [{"role": "user", "content": "I'm looking to generate a pet food ad for young families"}]})
print(result.model_dump(exclude_none=True))

In [0]:
# direct mode
result = AGENT.predict({"input": [], "custom_inputs": { "direct": { "prompt": "French Bulldog", "replicate_toggle": True } }})
print(result.model_dump(exclude_none=True))

## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

In [0]:
from config import DeployConfig

dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
agent_endpoint_name = getattr(cfg, f"agent_endpoint_name")
image_gen_model_path = getattr(cfg, f"image_gen_model_path").path

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from mlflow.models.resources import DatabricksServingEndpoint

resources = [
    DatabricksServingEndpoint(endpoint_name="databricks-claude-3-7-sonnett"),
    DatabricksServingEndpoint(endpoint_name=agent_endpoint_name),
]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            "databricks-openai", 
            "backoff"
        ],
        resources=resources
    )

## Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "I'm looking to generate a pet food ad for young families"}]},
    env_manager="uv",
)

## Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = f"{image_gen_model_path}_chat"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

## Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    endpoint_name = f"{agent_endpoint_name}_chat"
)